In [2]:
#  Practical Assignment: Data Preprocessing Pipeline - Lesson 3

# 1. Import the necessary libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("Successfully imported libraries")

Successfully imported libraries


In [15]:
# 1 - Load & Inspect Dataset - Car Prices
# === LOAD DATASET FROM EXCEL FILE
import pandas as pd  # Make sure pandas is imported

# Use pd.read_excel() instead of pd.read_csv() for Excel files
df = pd.read_excel("raw_car_dataset.csv.Xlsx")

# Alternative solution if the file is actually a CSV with wrong extension:
# df = pd.read_csv("raw_car_dataset.csv.Xlsx", encoding='latin-1')

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [16]:
# === INITIAL SNAPSHOT
print("\n===  INITIAL HEAD:")
print(df.head())


===  INITIAL HEAD:
   Price  Odometer_km     Doors  Accidents  Location    Year  Unnamed: 6
0    NaN         1500  137830.0        4.0         1    City        1998
1    NaN         4171       NaN        4.0         0   Rural        2016
2    NaN         5331  107302.0        4.0         0  Suburb        2014
3    NaN         1500  141838.0        4.0         1  Suburb        1999
4    NaN         1500       NaN        3.0         0    City        2022


In [17]:
print("\n===  INITIAL INFO:")
print(df.info())


===  INITIAL INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        0 non-null      float64
 1   Odometer_km  145 non-null    int64  
 2   Doors        138 non-null    float64
 3   Accidents    138 non-null    float64
 4   Location     145 non-null    int64  
 5   Year         140 non-null    object 
 6   Unnamed: 6   145 non-null    int64  
dtypes: float64(3), int64(3), object(1)
memory usage: 8.1+ KB
None


In [18]:
print("\n===  INITIAL SHAPE:")
print(df.shape)


===  INITIAL SHAPE:
(145, 7)


In [19]:
print("\n===  INITIAL MISSING VALUES:")
print(df.isnull().sum())


===  INITIAL MISSING VALUES:
Price          145
Odometer_km      0
Doors            7
Accidents        7
Location         0
Year             5
Unnamed: 6       0
dtype: int64


In [20]:
# === COUNT UNIQUE VALUES
print("\n===  COUNT UNIQUE VALUES:")
print(df.nunique())


===  COUNT UNIQUE VALUES:
Price            0
Odometer_km     55
Doors          132
Accidents        4
Location         4
Year             5
Unnamed: 6      26
dtype: int64


In [21]:
# === LOCATION COUNTS
print("\n===  LOCATION COUNTS:")
print(df["Location"].value_counts(dropna=False))


===  LOCATION COUNTS:
Location
0    76
1    40
2    23
3     6
Name: count, dtype: int64


In [23]:
# 2 - Clean Target Formatting (Price)  - remove currency/commas; ensure numeric; report dtype and skewness.
# First convert to string to handle any numeric values, then clean and convert back to float
df["Price"] = df["Price"].astype(str).str.replace("[$,]", "", regex=True).astype(float)
print("\n===  PRICE DATATYPE AND SKEWNESS:")
print("Price Datatype:", df["Price"].dtype)
print("Price Skewness:" ,df['Price'].skew())


===  PRICE DATATYPE AND SKEWNESS:
Price Datatype: float64
Price Skewness: nan


In [24]:
# 3 - Fix Category Errors before Imputation — normalize Location text, map typos (e.g., Subrb→Suburb), convert unknowns (e.g., “??”) to missing; recount including missing.
df['Location'] =df ["Location"].replace({"Subrb": "Suburb", "??": pd.NA})
print("\n===  LOCATION COUNTS AFTER FIXES:")
print(df["Location"].value_counts(dropna=False))


===  LOCATION COUNTS AFTER FIXES:
Location
0    76
1    40
2    23
3     6
Name: count, dtype: int64


In [25]:
# 4 -Impute Missing Values (justify choices) — Odometer_km→median; Doors/Accidents→mode; Location→mode; confirm post-imputation missing counts.
df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"] = df["Doors"].fillna(df["Doors"].mode()[0])
df["Accidents"] = df["Accidents"].fillna(df["Accidents"].mode()[0])
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])

print("\n===  MISSING VALUES AFTER IMPUTATION:")
print(df.isnull().sum())


===  MISSING VALUES AFTER IMPUTATION:
Price          145
Odometer_km      0
Doors            0
Accidents        0
Location         0
Year             5
Unnamed: 6       0
dtype: int64


In [26]:
# 5 - Remove Duplicates — report shape before/after and rows removed.
Before = df.shape
df= df.drop_duplicates()
After = df.shape
print("\n===  SHAPE BEFORE AND AFTER DUPLICATE REMOVAL:")
print("Before:", Before)
print("After:", After)


===  SHAPE BEFORE AND AFTER DUPLICATE REMOVAL:
Before: (145, 7)
After: (140, 7)


In [27]:
# 6 - Outliers (IQR capping) — compute bounds and clip for Price and Odometer_km; show a short summary after capping.
def iqr_function(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper
price_lower, price_upper = iqr_function(df["Price"])
odometer_lower, odometer_upper = iqr_function(df["Odometer_km"])
df["Price"] = df["Price"].clip(price_lower, price_upper)
df["Odometer_km"] = df["Odometer_km"].clip(odometer_lower, odometer_upper)

print("\n===  PRICE AND ODOMETER_KM SUMMARY AFTER CAPPING:")
print(df[["Price", "Odometer_km"]].describe())


===  PRICE AND ODOMETER_KM SUMMARY AFTER CAPPING:
       Price  Odometer_km
count    0.0   140.000000
mean     NaN  3175.456250
std      NaN  2601.848629
min      NaN  1500.000000
25%      NaN  1500.000000
50%      NaN  1500.000000
75%      NaN  4489.750000
max      NaN  8974.375000


In [28]:
# 7 - One-Hot Encode Categorical(s) — encode Location as 0/1 columns; list the new columns created.
df = pd.get_dummies(df, columns=["Location"], drop_first=False, dtype="int")
print("\n===  NEW COLUMNS AFTER ONE-HOT ENCODING LOCATION:")
print([col for col in df.columns if col.startswith("Location_")])


===  NEW COLUMNS AFTER ONE-HOT ENCODING LOCATION:
['Location_0', 'Location_1', 'Location_2', 'Location_3']


TypeError: unsupported operand type(s) for -: 'int' and 'str'